# Experiment 6 - Part 2: Metadata-Filtered Search

**Course:** HAAI302COM - Artificial Intelligence Laboratory

Pure vector search answers "what is most similar?". Real questions add conditions: "most similar *among papers from 2015 or later*", "only in computational linguistics", "not already reviewed". These conditions are **filters on the payload**. In this notebook we write filters, check what they do to the results, see why filtering must happen *inside* the index, and speed filters up with payload indexes.

Run notebook 01 first; it creates the `arxiv` collection that we use here.

## 0. Setup

In [1]:
import os, time, warnings
warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 170)

from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer

client = QdrantClient(url="http://localhost:6333")
NAME = "arxiv"
assert client.collection_exists(NAME), "run notebook 01 first"
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2"); model.max_seq_length = 128
print("points in", NAME, ":", client.count(NAME).count)

def embed(text):
    return model.encode(text, normalize_embeddings=True).tolist()

def search(text, k=5, flt=None, **kw):
    res = client.query_points(NAME, query=embed(text), query_filter=flt, limit=k, with_payload=True, **kw).points
    return pd.DataFrame([{"score": round(r.score, 3), "year": r.payload["year"], "cat": r.payload["primary_category"],
                          "title": r.payload["title"][:70]} for r in res])

points in arxiv : 11762


## 1. The filter language

A `Filter` combines **conditions** with three logical groups:

| Group | Meaning | SQL analogy |
|---|---|---|
| `must` | every condition must be true | `AND` |
| `should` | at least one condition must be true | `OR` |
| `must_not` | none of the conditions may be true | `AND NOT` |

Conditions are `FieldCondition(key=..., match=...)` for exact values and lists, or `range=...` for numbers.

| Condition | Example |
|---|---|
| `MatchValue(value="cs.CL")` | field equals the value |
| `MatchAny(any=["cs.CL", "cs.LG"])` | field is one of the values |
| `Range(gte=2015, lt=2020)` | number in an interval |
| `MatchText(text="neural")` | text contains the word (needs a text index, section 5) |

If the payload field is a list (like `categories`), a match succeeds when **any element** matches. We run the same query unfiltered and with four different filters (the query is deliberately broad, so that it matches papers from several fields and eras).

In [2]:
Q = "neural networks and learning"
F = models.FieldCondition
filters = {
    "no filter": None,
    "category = cs.LG": models.Filter(must=[F(key="primary_category", match=models.MatchValue(value="cs.LG"))]),
    "year < 2005": models.Filter(must=[F(key="year", range=models.Range(lt=2005))]),
    "cs.LG AND year >= 2015": models.Filter(must=[F(key="primary_category", match=models.MatchValue(value="cs.LG")), F(key="year", range=models.Range(gte=2015))]),
    "stat.ML OR cond-mat.dis-nn in categories": models.Filter(should=[F(key="categories", match=models.MatchValue(value="stat.ML")), F(key="categories", match=models.MatchValue(value="cond-mat.dis-nn"))]),
}
for name, flt in filters.items():
    print(f"--- {name}")
    print(search(Q, k=3, flt=flt).to_string(index=False))

--- no filter
 score  year   cat                                                                  title
 0.530  2021 cs.LG Understanding Dynamics of Nonlinear Representation Learning and Its Ap
 0.512  2020 cs.CE A Physics-Guided Neural Network Framework for Elastic Plates: Comparis
 0.490  2018 cs.LG Learning Neuron Non-Linearities with Kernel-Based Deep Neural Networks
--- category = cs.LG
 score  year   cat                                                                  title
 0.530  2021 cs.LG Understanding Dynamics of Nonlinear Representation Learning and Its Ap
 0.490  2018 cs.LG Learning Neuron Non-Linearities with Kernel-Based Deep Neural Networks
 0.469  2019 cs.LG Deep Learning Theory Review: An Optimal Control and Dynamical Systems 
--- year < 2005
 score  year             cat                                                                  title
 0.406  1996          cmp-lg                                      Unsupervised Language Acquisition
 0.326  2002 cond-mat.dis-nn 

 score  year   cat                                                                  title
 0.530  2021 cs.LG Understanding Dynamics of Nonlinear Representation Learning and Its Ap
 0.490  2018 cs.LG Learning Neuron Non-Linearities with Kernel-Based Deep Neural Networks
 0.469  2019 cs.LG Deep Learning Theory Review: An Optimal Control and Dynamical Systems 
--- stat.ML OR cond-mat.dis-nn in categories
 score  year   cat                                                                  title
 0.530  2021 cs.LG Understanding Dynamics of Nonlinear Representation Learning and Its Ap
 0.490  2018 cs.LG Learning Neuron Non-Linearities with Kernel-Based Deep Neural Networks
 0.469  2019 cs.LG Deep Learning Theory Review: An Optimal Control and Dynamical Systems 


**What to notice:** one query, very different answers. Unfiltered, the third hit is a physics-guided neural network from `cs.CE`; with `category = cs.LG` that paper disappears and a 2019 `cs.LG` paper moves up. `year < 2005` gives a completely different list (language acquisition from 1996, associative memory in recurrent networks from 2002, quasar candidate selection from 2004) with much lower scores (0.30 to 0.41), because few early papers relate to the query. The last two filters return the same top three as `cs.LG` alone: those papers already satisfy them, so the extra conditions change nothing here. Note that the surviving papers keep their scores (the top hit is 0.530 each time): a filter only shrinks the candidate pool, it does not re-score.

## 2. Selectivity: how much of the data does a filter keep?

The **selectivity** of a filter is the fraction of points that pass. It decides how hard the search is: a filter that keeps half the data is easy, one that keeps 0.1 percent is hard for a graph index. `count` with a filter returns the number of matching points.

In [3]:
total = client.count(NAME).count
rows = []
for name, flt in filters.items():
    if flt is None: continue
    n = client.count(NAME, count_filter=flt, exact=True).count
    rows.append({"filter": name, "matching": n, "selectivity": f"{n / total:.2%}"})
pd.DataFrame(rows)

,filter,matching,selectivity
0,category = cs.LG,271,2.30%
1,year < 2005,1788,15.20%
2,cs.LG AND year >= 2015,255,2.17%
3,stat.ML OR cond-mat.dis-nn in categories,411,3.49%


**What to notice:** `category = cs.LG` keeps 271 papers (2.3 percent), `year < 2005` keeps 15 percent, and the `should` filter 411 papers (3.5 percent). Adding `year >= 2015` to `cs.LG` barely changes the count (271 to 255), because most cs.LG papers are recent anyway, so a combined filter is not always much narrower than its parts. Selectivity depends on the data, so measure it with `count` before trusting a filter in production.

## 3. Why filtering must happen inside the index

A naive design searches first and filters afterwards (**post-filtering**): take the top 10 vectors, then throw away those that fail the condition. The result can contain fewer than 10 items, or none. Qdrant instead evaluates the condition *during* the graph traversal (**filterable HNSW**), so it keeps looking until it has 10 matching results.

We test this on 100 queries (the titles of random papers) with a restrictive filter: papers from before the year 2000.

In [4]:
rng = np.random.default_rng(0)
sample = client.scroll(NAME, limit=100, with_payload=True, with_vectors=True, offset=None)[0]
qvecs = [r.vector for r in sample]
flt = models.Filter(must=[F(key="year", range=models.Range(lt=2000))])
print("filter passes", f"{client.count(NAME, count_filter=flt, exact=True).count / total:.1%}", "of the collection")

K = 10
post, inside = [], []
for v in qvecs:
    top = client.query_points(NAME, query=v, limit=K, with_payload=True).points           # search first ...
    post.append(sum(p.payload["year"] < 2000 for p in top))                                 # ... then filter
    inside.append(len(client.query_points(NAME, query=v, query_filter=flt, limit=K).points))
pd.DataFrame({"post-filter": pd.Series(post).describe(), "filter inside the index": pd.Series(inside).describe()}).loc[["mean", "min", "max"]].round(2)

filter passes 6.0% of the collection


,post-filter,filter inside the index
mean,0.63,10.0
min,0.00,10.0
max,3.00,10.0


**What to notice:** post-filtering returns on average only 0.63 of the 10 requested results, and for some queries none at all (minimum 0, maximum 3). This is the arithmetic you would expect: the filter keeps 6 percent of the collection, so about 0.6 of the unfiltered top 10 survive. Filtering inside the index returns exactly 10 results for every query. The rarer the condition, the worse post-filtering gets, which is why a vector database must support filters natively.

Is the filtered HNSW search also *accurate*? We compare it with an **exact** filtered search (`SearchParams(exact=True)` scans every matching point) and measure how many of the exact top 10 it finds.

In [5]:
def recall_vs_exact(flt, vectors, hnsw_ef=None):
    got = 0
    for v in vectors:
        approx = {p.id for p in client.query_points(NAME, query=v, query_filter=flt, limit=K, search_params=models.SearchParams(hnsw_ef=hnsw_ef)).points}
        exact = {p.id for p in client.query_points(NAME, query=v, query_filter=flt, limit=K, search_params=models.SearchParams(exact=True)).points}
        got += len(approx & exact) / max(len(exact), 1)
    return got / len(vectors)

print("recall@10 of filtered HNSW (year < 2000):", round(recall_vs_exact(flt, qvecs[:50]), 3))

recall@10 of filtered HNSW (year < 2000): 1.0


**What to notice:** recall is 1.0. That is expected here: this filter keeps only about 700 points, far below the `full_scan_threshold` we saw in Notebook 01, so Qdrant scans the matching points exactly instead of walking the graph. For broad filters it uses the graph with the filter applied during traversal, where recall can fall; Notebook 03 measures that on a larger collection.

## 4. Payload indexes

By default a filter must read each candidate's payload, which is slow. A **payload index** on a field lets Qdrant answer "which points have year >= 2015" from a data structure and even plan the search around it: for a very selective filter it can skip the vector graph and score only the few matching points. Index types: `keyword` (exact strings), `integer`, `float`, `bool`, `text` (full text), `datetime`, `geo`.

We time 100 filtered queries before and after creating indexes on `primary_category`, `year` and `has_journal_ref`.

In [6]:
selective = models.Filter(must=[F(key="primary_category", match=models.MatchValue(value="cs.CL")), F(key="year", range=models.Range(gte=2018))])

def time_filter(flt, vectors, reps=2):
    best = 1e9
    for _ in range(reps):
        t0 = time.time()
        for v in vectors:
            client.query_points(NAME, query=v, query_filter=flt, limit=K)
        best = min(best, (time.time() - t0) / len(vectors) * 1000)
    return best

for field in ["primary_category", "year", "has_journal_ref", "title"]:             # drop indexes left by an earlier run, so this cell is repeatable
    try:
        client.delete_payload_index(NAME, field_name=field, wait=True)
    except Exception:
        pass
print("payload indexes now:", list(client.get_collection(NAME).payload_schema))
n_sel = client.count(NAME, count_filter=selective, exact=True).count
no_index = time_filter(selective, qvecs)
print("no filter at all:", round(time_filter(None, qvecs), 2), "ms per query")
print("matching points:", n_sel, f"({n_sel / total:.2%})", "| without payload index:", round(no_index, 2), "ms per query")

payload indexes now: []


no filter at all: 7.31 ms per query
matching points: 115 (0.98%) | without payload index: 28.0 ms per query


In [7]:
for field, schema in [("primary_category", models.PayloadSchemaType.KEYWORD), ("year", models.PayloadSchemaType.INTEGER),
                      ("has_journal_ref", models.PayloadSchemaType.BOOL)]:
    client.create_payload_index(NAME, field_name=field, field_schema=schema, wait=True)

print({k: v.data_type.value for k, v in client.get_collection(NAME).payload_schema.items()})
with_index = time_filter(selective, qvecs)
print("with payload indexes:", round(with_index, 2), "ms per query | speed-up:", round(no_index / with_index, 1), "x")

{'has_journal_ref': 'bool', 'year': 'integer', 'primary_category': 'keyword'}


with payload indexes: 5.75 ms per query | speed-up: 4.9 x


**What to notice:** without payload indexes, the selective filter (115 papers, 0.98 percent of the collection) costs about 28 ms per query, almost four times slower than unfiltered search (about 7 ms), because Qdrant has to read candidates' payloads to test the condition. With payload indexes it takes about 6 ms, roughly five times faster, and even faster than no filter at all: the index shows that only 115 points qualify, so Qdrant scores just those instead of traversing the graph. Rule of thumb: create a payload index on every field you filter, facet or group by.

## 5. Full-text filters

For free-text conditions ("title contains the word *quantum*") create a `text` index with a tokenizer. Then `MatchText` filters on words, which can be combined with vector search: semantic ranking among documents that literally contain a word.

In [8]:
client.create_payload_index(NAME, field_name="title", wait=True,
    field_schema=models.TextIndexParams(type="text", tokenizer=models.TokenizerType.WORD, lowercase=True, min_token_len=3))

has_word = models.Filter(must=[F(key="title", match=models.MatchText(text="learning"))])
print("titles containing 'learning':", client.count(NAME, count_filter=has_word, exact=True).count)
search("methods for training without labels", k=4, flt=has_word)

titles containing 'learning': 321


,score,year,cat,title
0,0.493,2019,cs.LG,Self-Supervised Generalisation with Meta Auxiliary Learning
1,0.447,2019,cs.CV,Retro-Actions: Learning 'Close' by Time-Reversing 'Open' Videos
2,0.440,2020,cs.CV,Attention-Aware Noisy Label Learning for Image Classification
3,0.420,2020,cs.CL,Single-/Multi-Source Cross-Lingual NER via Teacher-Student Learning on


**What to notice:** 321 titles contain the word "learning" (2.7 percent). The semantic query "methods for training without labels" is ranked only among those, and the best score is modest (0.49), because the most semantically similar papers may not contain the literal word. A text filter is exact and lexical (here without stemming, so "learn" would not match "learning"). It is useful for must-contain constraints such as product codes or names, combined with meaning-based ranking.

## 6. Facets and grouped results

Two features that matter for search interfaces:

- **Facets** count how many points fall under each value of a field (the "narrow by category" sidebar). They need a payload index on the field.
- **Grouped search** returns the best hits *per group*, which diversifies results instead of returning ten near-identical papers from one field.

In [9]:
fac = client.facet(NAME, key="primary_category", limit=8)
pd.DataFrame([{"category": h.value, "papers": h.count} for h in fac.hits])

,category,papers
0,hep-ph,709
1,hep-th,586
2,astro-ph,571
3,quant-ph,534
4,cond-mat.mes-hall,332
5,gr-qc,321
6,cs.CV,284
7,cs.LG,271


In [10]:
grp = client.query_points_groups(NAME, query=embed("deep learning for images and language"), group_by="primary_category", limit=4, group_size=2, with_payload=True)
rows = [{"group": g.id, "score": round(h.score, 3), "year": h.payload["year"], "title": h.payload["title"][:65]} for g in grp.groups for h in g.hits]
pd.DataFrame(rows)

,group,score,year,title
0,cs.CL,0.526,2017,Visually grounded learning of keyword prediction from untranscrib
1,cs.CL,0.489,2018,Latent Variable Model for Multi-modal Translation
2,cs.CV,0.508,2017,FOIL it! Find One mismatch between Image and Language caption
3,cs.CV,0.487,2021,MaIL: A Unified Mask-Image-Language Trimodal Network for Referrin
4,eess.IV,0.503,2020,Quantitative Digital Microscopy with Deep Learning
5,eess.IV,0.438,2021,Artifact Reduction in Fundus Imaging using Cycle Consistent Adver
6,cs.LG,0.481,2016,Learning Hand-Eye Coordination for Robotic Grasping with Deep Lea
7,cs.LG,0.463,2019,HyperLearn: A Distributed Approach for Representation Learning in


**What to notice:** the facet counts match the category counts we computed in pandas in Notebook 01 (hep-ph 709, hep-th 586, ...), but computed by the database. The grouped search returned two papers from each of four different fields (cs.CL, cs.CV, eess.IV, cs.LG), ordered by each group's best score, whereas plain top-8 would likely give several papers from the single closest field. Grouping is a cheap way to diversify results.

## Summary

| Question | Answer |
|---|---|
| How do I express conditions? | `Filter(must / should / must_not)` of `FieldCondition` with `MatchValue`, `MatchAny`, `Range`, `MatchText` |
| Why not filter after search? | It returns too few results; Qdrant filters inside the graph search |
| Do filters slow search or lose accuracy? | Check with `exact=True` on your own filters; selective filters are the risky case |
| How do I speed filters up? | `create_payload_index` on every field you filter, facet or group by |
| How do I build a search UI? | facets for sidebars, groups for diverse results |

**Next:** Notebook 03 benchmarks index configurations (HNSW settings, quantisation) for latency and recall, and shows the same operations in Pinecone.